# HotpotQA Scan Cache Explore

这个 notebook 用来：

- 按当前项目的方式读取 HotpotQA scan cache
- 查看 cache 中所有 span 的出现次数
- 为后续构造新的数据子集做准备


In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")

Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
from collections import Counter
from pathlib import Path
import pickle

import pandas as pd

from text_processing import normalize_text

In [3]:
HOTPOT_SCAN_STORE_PATH = Path("hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl")
pd.set_option("display.max_colwidth", None)


def load_hotpot_scan_store(scan_store_path: Path = HOTPOT_SCAN_STORE_PATH):
    if not scan_store_path.exists():
        raise FileNotFoundError(
            f"HotpotQA scan store not found at {scan_store_path}. Please prepare the scan cache first."
        )

    with scan_store_path.open("rb") as handle:
        store = pickle.load(handle)

    print(f"Loaded scan-only store from {scan_store_path}")
    print(store["stats"])
    return store


def lookup_records(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


embedding_store = load_hotpot_scan_store()

Loaded scan-only store from hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl
{'num_documents': 66581, 'num_unique_terms': 634624, 'num_phrase_occurrences': 1007780, 'num_token_occurrences': 584999, 'num_total_occurrences': 1592779}


In [4]:
def build_span_frequency_dataframe(store):
    rows = []

    for term, records in store["index"].items():
        kind_counter = Counter(record["kind"] for record in records)
        rows.append(
            {
                "term": term,
                "count": len(records),
                "phrase_count": int(kind_counter.get("phrase", 0)),
                "token_count": int(kind_counter.get("token", 0)),
                "num_unique_documents": len({record["document_idx"] for record in records}),
            }
        )

    df = pd.DataFrame(rows)
    df = df.sort_values(
        by=["count", "num_unique_documents", "term"],
        ascending=[False, False, True],
        ignore_index=True,
    )
    df.insert(0, "rank", range(1, len(df) + 1))
    return df


span_frequency_df = build_span_frequency_dataframe(embedding_store)
print(f"Total unique spans: {len(span_frequency_df)}")
display(span_frequency_df)


Total unique spans: 634624


,rank,term,count,phrase_count,token_count,num_unique_documents
0,1,film,6196,0,6196,4199
1,2,united states,5295,5295,0,4822
2,3,song,4721,5,4716,2481
3,4,album,4352,1,4351,2776
4,5,time,3928,19,3909,3515
...,...,...,...,...,...,...
634619,634620,𐌱𐌴𐌽𐌴𐌳𐌹𐌺𐍄,1,0,1,1
634620,634621,𐎢𐎺𐎧𐏁𐎫𐎼,1,0,1,1
634621,634622,𐎨𐎡𐏁𐎱𐎡𐏁 cišpiš,1,1,0,1
634622,634623,𒊑𒅀,1,0,1,1


In [5]:
TOP_K = 100
span_frequency_df.head(TOP_K)

,rank,term,count,phrase_count,token_count,num_unique_documents
0,1,film,6196,0,6196,4199
1,2,united states,5295,5295,0,4822
2,3,song,4721,5,4716,2481
3,4,album,4352,1,4351,2776
4,5,time,3928,19,3909,3515
...,...,...,...,...,...,...
95,96,ireland,732,697,35,546
96,97,billboard,726,5,721,595
97,98,character,723,0,723,610
98,99,actor,722,0,722,672


In [7]:
PRINT_TOP_K = 50

top_terms = span_frequency_df.head(PRINT_TOP_K)["term"].tolist()
print(f"Top {PRINT_TOP_K} terms:")
print("[")
for term in top_terms:
    print(f"    {term!r},")
print("]")

Top 50 terms:
[
    'film',
    'united states',
    'song',
    'album',
    'time',
    'series',
    'band',
    'city',
    'member',
    'single',
    'number',
    'team',
    'world',
    'place',
    'work',
    'season',
    'music',
    'group',
    'company',
    'game',
    'california',
    'new york',
    'england',
    'state',
    'title',
    'role',
    'members',
    'people',
    'films',
    'book',
    'year',
    'town',
    'career',
    'australia',
    'area',
    'story',
    'death',
    'canada',
    'president',
    'son',
    'life',
    'history',
    'songs',
    'population',
    'US',
    'director',
    'new york city',
    'english',
    'london',
    'novel',
]
